# M044_2024_12_04_09_30

Session: M044_2024_12_04_09_30

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns
import pyaldata as pyal

from tools.dsp.preprocessing import preprocess
from tools.decoding import decodeTools as decode

import tools.viz.rasters as rt
from tools.params import Params
from tools.viz.dimensionality import plot_VAF
from tools.dataTools import get_data_array
from tools.viz.rasters import plot_heatmap_raster

from tools.dimensionality.participation import participation_ratio, pca_pr
from tools.params import colors
from tools.dimensionality.cca import canoncorr
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score




In [ ]:
# TODO: add example data to the repo and run on that
data_dir = "/data/bnd-data/raw/M044/M044_2024_12_04_09_30"
fname0 = os.path.join(data_dir, "M044_2024_12_04_09_30_pyaldata_0.mat")
fname1 = os.path.join(data_dir, "M044_2024_12_04_09_30_pyaldata_1.mat")

In [ ]:
df0 = pyal.mat2dataframe(fname0, shift_idx_fields=True)
df1 = pyal.mat2dataframe(fname1, shift_idx_fields=True)
df = pd.concat([df0, df1], ignore_index=True)

In [ ]:
df.head()

## Preprocessing from utils

In [ ]:
df_ = preprocess(df, only_trials=False)
areas = ["M1", "Dls"]
df_["M1_rates"] = [df_["all_imec0_rates"][i][:,300:] for i in range(len(df_))]
df_["Dls_rates"] = [df_["all_imec0_rates"][i][:,0:300] for i in range(len(df_))]

## Plotting

In [ ]:
df_ = pyal.select_trials(df_, "idx_trial_end > 30365")  # Remove first 5 minutes because the switch was off

In [ ]:
df_.head()

In [ ]:
df_.idx_sol_on

- Trial 90 has some nice spikes

In [ ]:
areas = ["M1", "Dls"]
df_["M1_rates"] = [df_["all_rates"][i][:,300:] for i in range(len(df_))]
df_["Dls_rates"] = [df_["all_rates"][i][:,0:300] for i in range(len(df_))]

In [ ]:
fig, axes = plt.subplots(1, figsize=(15, 5), sharey=True)

# axes.imshow(rates.T, aspect="auto")
area="Dls"
plot_heatmap_raster(pyal.select_trials(df_, df_.trial_name == 'trial')[50:75], area=area,ax=axes, show=False, add_sol_onset=True)
plt.show()

# rates.shape

### Decoding

In [ ]:
perturb_epoch = pyal.generate_epoch_fun(
        start_point_name="idx_sol_on",
        rel_start=int(0 / Params.BIN_SIZE),
        rel_end=int(1 / Params.BIN_SIZE),
    )

df_trials = pyal.select_trials(df_, "trial_name == 'trial'")
df_trials = pyal.select_trials(df_trials, "idx_trial_end > 30365")  # Remove first 5 minutes because the switch was off
df_trials = pyal.restrict_to_interval(df_trials, epoch_fun=perturb_epoch)


fig, ax = plt.subplots()
within_results = decode.within_decoding(cat = "sol_level_id", ax = ax,  allDFs = [df_trials], area = "M1", n_components = 10, epoch = perturb_epoch, model = "pca")

In [ ]:
area = 'M1'
n_components = 10

perturb_epoch = pyal.generate_epoch_fun(
        start_point_name="idx_sol_on",
        rel_start=int(-2 / Params.BIN_SIZE),
        rel_end=int(1 / Params.BIN_SIZE),
    )

df_trials = pyal.select_trials(df_, "trial_name == 'trial'")
df_trials = pyal.select_trials(df_trials, "idx_trial_end > 30365")  # Remove first 5 minutes because the switch was off
# df_trials = pyal.restrict_to_interval(df_trials, epoch_fun=perturb_epoch)



rates = np.concatenate(df_trials[area + "_rates"].values, axis=0)  # Shape: (239 trials, 15 timepoints, 87 units)

# Fit PCA model
rates_model = PCA(n_components=n_components, svd_solver="full").fit(rates)

# Apply PCA to the dataframe
df_trials = pyal.apply_dim_reduce_model(df_trials, rates_model, area + "_rates", "_pca")



In [ ]:
df_trials_upper = pyal.select_trials(df_trials, df_trials['sol_level_id'] == 0)
df_trials_lower = pyal.select_trials(df_trials, df_trials['sol_level_id'] == 1)


In [ ]:
latents_upper = pyal.get_sig_by_trial(df_trials_upper, "_pca")
latents_upper = np.mean(latents_upper, axis=2)[:, :n_components]  # Reduce to first 3 PCA components

latents_lower = pyal.get_sig_by_trial(df_trials_lower, "_pca")
latents_lower = np.mean(latents_lower, axis=2)[:, :n_components]  # Reduce to first 3 PCA components


In [ ]:
latents_upper.shape

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': '3d'})

ax.plot(latents_upper[:, 0], latents_upper[:, 1], latents_upper[:, 2], linewidth=2)
ax.plot(latents_lower[:, 0], latents_lower[:, 1], latents_lower[:, 2], linewidth=2)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.view_init(30, 60)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(latents_upper[:, 0], latents_upper[:, 1], linewidth=2)
ax.plot(latents_lower[:, 0],  latents_lower[:, 1], linewidth=2)
# ax.view_init(60, 0)


In [ ]:
from sklearn.metrics import confusion_matrix


AllData = get_data_array(
    [df_trials],
    "sol_level_id",
    area='M1',
    model='pca',
    n_components=10,
)

AllData = AllData[0, ...]
n_targets, n_trial, n_time, n_comp = AllData.shape

# print(AllData.shape)
# resizing

target_ids = np.unique(df_trials["sol_level_id"])

X = np.squeeze(AllData.reshape((-1, n_time* n_comp)))
AllTar = np.repeat(target_ids, n_trial)
AllTar = np.array(AllTar, dtype=int).flatten()

y_pred = cross_val_predict(GaussianNB(), X, AllTar, cv=5)
conf_mat = confusion_matrix(AllTar, y_pred, labels=target_ids)



In [ ]:
conf_mat